# Dermoscopic Image Cancer Detection

Binary image classification with a custom CNN and transfer learning models.

In [ ]:
import os
import glob
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score, auc
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = "cnn_cancer_detector"
CLASS_NAMES = ["benign", "malignant"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## 1. Dataset

In [ ]:
EXPECTED_COUNTS = {
    "benign": 1800,
    "malignant": 1497
}

CLASS_NAMES = ["benign", "malignant"]
DATA_DIR = "."


def build_dataframe(data_dir, class_names):
    """Build a dataframe containing image paths and class labels."""
    records = []

    for label_idx, cls in enumerate(class_names):
        cls_dir = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_dir):
            print(
                f"WARNING: folder not found -> "
                f"{os.path.abspath(cls_dir)}"
            )
            continue

        image_files = []
        for ext in ("*.jpg", "*.jpeg", "*.png"):
            image_files.extend(glob.glob(os.path.join(cls_dir, ext)))

        for filepath in sorted(image_files):
            records.append({
                "filepath": filepath,
                "label_name": cls,
                "label": label_idx
            })

    return pd.DataFrame(records)


df = build_dataframe(DATA_DIR, CLASS_NAMES)
print(f"Total images found: {len(df)}")

if len(df) == 0:
    print("\nERROR: No images were found.")
    print("Current working directory:")
    print(os.getcwd())
else:
    found_counts = df["label_name"].value_counts().to_dict()
    print("\nImage counts:")
    for cls, expected in EXPECTED_COUNTS.items():
        actual = found_counts.get(cls, 0)
        status = "OK" if actual == expected else "MISMATCH"
        print(
            f"  {cls:10s} "
            f"expected={expected:5d}  "
            f"found={actual:5d}  "
            f"[{status}]"
        )

    print("\nFirst 5 images:")
    display(df.head())

### 2.3 Class Distribution

In [ ]:
class_counts = df["label_name"].value_counts()
print(class_counts)

plt.figure(figsize=(5, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio (majority:minority): {imbalance_ratio:.2f} : 1")


### 2.4 Sample Images

In [ ]:
def show_samples(df, class_names, n_per_class=4):
    fig, axes = plt.subplots(len(class_names), n_per_class, figsize=(n_per_class * 3, len(class_names) * 3))
    for row, cls in enumerate(class_names):
        samples = df[df["label_name"] == cls].sample(n=min(n_per_class, len(df[df["label_name"] == cls])), random_state=SEED)
        for col, (_, rec) in enumerate(samples.iterrows()):
            img = cv2.cvtColor(cv2.imread(rec["filepath"]), cv2.COLOR_BGR2RGB)
            ax = axes[row, col] if len(class_names) > 1 else axes[col]
            ax.imshow(img)
            ax.set_title(cls)
            ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(df, CLASS_NAMES)


## 2. Data Preprocessing

In [ ]:
def clean_dataframe(df):
    valid_rows = []
    for _, rec in df.iterrows():
        try:
            img = cv2.imread(rec["filepath"])
            if img is None or img.size == 0:
                continue
            valid_rows.append(rec)
        except Exception:
            continue
    cleaned = pd.DataFrame(valid_rows).reset_index(drop=True)
    print(f"Removed {len(df) - len(cleaned)} unreadable/corrupted images")
    return cleaned

df = clean_dataframe(df)

import hashlib

def file_hash(fp, block_size=65536):
    h = hashlib.md5()
    with open(fp, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

df["hash"] = df["filepath"].apply(file_hash)
before = len(df)
df = df.drop_duplicates(subset="hash").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate images")
df = df.drop(columns=["hash"])

### 3.2 Resizing & 3.3 Normalization
All images are resized to a fixed input size (224×224, matching the pretrained backbones) and pixel
values are scaled to the [0, 1] range. `ImageDataGenerator` handles both automatically at load time.

### 3.4 Data Augmentation
Because medical imaging datasets are often small and imbalanced, augmentation (rotation, flips, zoom,
shifts, brightness) is applied to the **training set only** to improve generalization and mitigate
overfitting.

### 3.5 Train / Validation / Test Split
An 70% / 15% / 15% stratified split preserves class proportions across all three sets.


In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

train_df["label_name"] = train_df["label_name"].astype(str)
val_df["label_name"] = val_df["label_name"].astype(str)
test_df["label_name"] = test_df["label_name"].astype(str)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=25,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode="nearest",
)
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_dataframe(
    train_df, x_col="filepath", y_col="label_name",
    target_size=IMG_SIZE, class_mode="binary",
    batch_size=BATCH_SIZE, shuffle=True, seed=SEED,
)
val_gen = val_test_datagen.flow_from_dataframe(
    val_df, x_col="filepath", y_col="label_name",
    target_size=IMG_SIZE, class_mode="binary",
    batch_size=BATCH_SIZE, shuffle=False,
)
test_gen = val_test_datagen.flow_from_dataframe(
    test_df, x_col="filepath", y_col="label_name",
    target_size=IMG_SIZE, class_mode="binary",
    batch_size=BATCH_SIZE, shuffle=False,
)

print("Class indices:", train_gen.class_indices)

from sklearn.utils.class_weight import compute_class_weight
class_weights_arr = compute_class_weight(
    class_weight="balanced", classes=np.unique(train_df["label"]), y=train_df["label"]
)
class_weights = dict(enumerate(class_weights_arr))
print("Class weights:", class_weights)

## 3. Exploratory Data Analysis

In [ ]:
sample_df = df.sample(n=min(300, len(df)), random_state=SEED)
dims = []
for fp in sample_df["filepath"]:
    img = cv2.imread(fp)
    if img is not None:
        dims.append(img.shape[:2])
dims_df = pd.DataFrame(dims, columns=["height", "width"])
print(dims_df.describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(dims_df["height"], ax=axes[0], kde=True)
axes[0].set_title("Image Height Distribution")
sns.histplot(dims_df["width"], ax=axes[1], kde=True)
axes[1].set_title("Image Width Distribution")
plt.tight_layout()
plt.show()

def mean_intensity(fp):
    img = cv2.imread(fp)
    return img.mean() if img is not None else np.nan

sample_df = sample_df.copy()
sample_df["mean_intensity"] = sample_df["filepath"].apply(mean_intensity)

plt.figure(figsize=(6, 4))
sns.boxplot(data=sample_df, x="label_name", y="mean_intensity", palette="viridis")
plt.title("Mean Pixel Intensity by Class")
plt.show()

## 4. Model Development

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ], name="custom_cnn")
    return model

custom_cnn = build_custom_cnn(input_shape=IMG_SIZE + (3,))
custom_cnn.summary()


### 5.2 Model Compilation

In [ ]:
custom_cnn.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)

cb_list = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    callbacks.ModelCheckpoint("best_custom_cnn.keras", monitor="val_auc", mode="max", save_best_only=True),
]

### 5.3 Training

In [ ]:
history_cnn = custom_cnn.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=cb_list,
)

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title(f"{title} - Loss")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_cnn, "Custom CNN")

## 5. Transfer Learning

In [ ]:
def build_transfer_model(base_model_fn, input_shape=(224, 224, 3), fine_tune_at=None):
    base_model = base_model_fn(include_top=False, weights="imagenet", input_shape=input_shape)
    base_model.trainable = False  # phase 1: frozen backbone

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = models.Model(inputs, outputs)
    return model, base_model

mobilenet_model, mobilenet_base = build_transfer_model(MobileNetV2, IMG_SIZE + (3,))
mobilenet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)
mobilenet_model.summary()

In [ ]:
# Phase 1: train the classification head only
history_mobilenet_head = mobilenet_model.fit(
    train_gen, validation_data=val_gen, epochs=10,
    class_weight=class_weights,
    callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True)],
)

# Phase 2: unfreeze top layers of the backbone and fine-tune at a low LR
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)

history_mobilenet_ft = mobilenet_model.fit(
    train_gen, validation_data=val_gen, epochs=15,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
        callbacks.ModelCheckpoint("best_mobilenet.keras", monitor="val_auc", mode="max", save_best_only=True),
    ],
)

### 6.2 EfficientNetB0

In [ ]:
effnet_model, effnet_base = build_transfer_model(EfficientNetB0, IMG_SIZE + (3,))
effnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)

# Phase 1: head only
history_effnet_head = effnet_model.fit(
    train_gen, validation_data=val_gen, epochs=10,
    class_weight=class_weights,
    callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True)],
)

# Phase 2: fine-tune top layers
effnet_base.trainable = True
for layer in effnet_base.layers[:-30]:
    layer.trainable = False

effnet_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")],
)

history_effnet_ft = effnet_model.fit(
    train_gen, validation_data=val_gen, epochs=15,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
        callbacks.ModelCheckpoint("best_effnet.keras", monitor="val_auc", mode="max", save_best_only=True),
    ],
)

## 6. Model Evaluation

In [ ]:
def evaluate_model(model, test_gen, model_name):
    test_gen.reset()
    y_true = test_gen.classes
    y_prob = model.predict(test_gen, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)

    print(f"=== {model_name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4.5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show()

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure(figsize=(4.5, 4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {model_name}")
    plt.legend()
    plt.show()

    return {"model": model_name, "accuracy": acc, "precision": prec,
            "recall": rec, "f1": f1, "roc_auc": roc_auc}

results_cnn = evaluate_model(custom_cnn, test_gen, "Custom CNN")

In [ ]:
results_mobilenet = evaluate_model(mobilenet_model, test_gen, "MobileNetV2 (fine-tuned)")

In [ ]:
results_effnet = evaluate_model(effnet_model, test_gen, "EfficientNetB0 (fine-tuned)")

## 7. Model Comparison

In [ ]:
comparison_df = pd.DataFrame([results_cnn, results_mobilenet, results_effnet]).set_index("model")
display(comparison_df.style.highlight_max(axis=0, color="lightgreen"))

comparison_df.plot(kind="bar", figsize=(10, 5))
plt.title("Model Comparison Across Metrics")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

best_model_name = comparison_df["roc_auc"].idxmax()
print(f"Best model by ROC-AUC: {best_model_name}")

## 8. Prediction on New Images

In [ ]:
def predict_image(model, image_path, img_size=IMG_SIZE, class_names=CLASS_NAMES, threshold=0.5):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, img_size)
    img_norm = img_resized.astype("float32") / 255.0
    img_batch = np.expand_dims(img_norm, axis=0)

    prob_malignant = model.predict(img_batch, verbose=0)[0][0]
    predicted_label = class_names[int(prob_malignant >= threshold)]

    plt.figure(figsize=(4, 4))
    plt.imshow(img_resized)
    plt.title(f"Prediction: {predicted_label} (p_malignant={prob_malignant:.3f})")
    plt.axis("off")
    plt.show()

    return predicted_label, float(prob_malignant)

# Example usage (replace with a real path to a new, unseen image):
# predict_image(mobilenet_model, "path/to/new_image.jpg")
